In [ ]:
import kagglehub
path = kagglehub.dataset_download("garciacoding/waste-classification")
print("Dataset path:", path)


100%|██████████| 32.6M/32.6M [00:00<00:00, 53.4MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/garciacoding/waste-classification/versions/2


In [ ]:
import os
train_candidate = os.path.join(path, "DATASET", "TRAIN")
test_candidate  = os.path.join(path, "DATASET", "TEST")

if os.path.exists(train_candidate):
    image_root = train_candidate
elif os.path.exists(test_candidate):
    image_root = test_candidate
else:

    IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    def looks_like_class_folder(root):
        if not os.path.isdir(root):
            return False
        subs = [os.path.join(root, d) for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
        if len(subs) < 2:
            return False
        for s in subs:
            files = os.listdir(s)
            if any(f.lower().endswith(IMG_EXTS) for f in files):
                return True
        return False

    def find_image_root(dataset_path):
        if looks_like_class_folder(dataset_path):
            return dataset_path
        for d in os.listdir(dataset_path):
            cand = os.path.join(dataset_path, d)
            if looks_like_class_folder(cand):
                return cand
        for root, dirs, files in os.walk(dataset_path):
            if looks_like_class_folder(root):
                return root
        raise ValueError("Could not find a folder with class subfolders containing images.")

    image_root = find_image_root(path)

print("Using image_root:", image_root)
print("Classes:", [d for d in os.listdir(image_root) if os.path.isdir(os.path.join(image_root, d))])


Using image_root: /root/.cache/kagglehub/datasets/garciacoding/waste-classification/versions/2/DATASET/TRAIN
Classes: ['METAL', 'PLASTIC']


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

np.random.seed(42)
tf.random.set_seed(42)

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
VAL_SPLIT = 0.2
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    image_root,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset="training",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    image_root,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset="validation",
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("num_classes:", num_classes)
print("class_names:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)


Found 1488 files belonging to 2 classes.
Using 1191 files for training.
Found 1488 files belonging to 2 classes.
Using 297 files for validation.
num_classes: 2
class_names: ['METAL', 'PLASTIC']


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])


In [ ]:
def build_transfer_model(base_model, preprocess_fn, num_classes):
    inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = data_augmentation(inputs)
    x = layers.Lambda(preprocess_fn)(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(256, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)


In [ ]:
import time

def train_and_finetune(model_name, BaseClass, preprocess_fn,
                       epochs_frozen=5, epochs_ft=5,
                       lr_frozen=1e-3, lr_ft=1e-5,
                       unfreeze_last_fraction=0.2):

    base = BaseClass(weights="imagenet", include_top=False,
                     input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base.trainable = False

    model = build_transfer_model(base, preprocess_fn, num_classes)

    model.compile(
        optimizer=keras.optimizers.Adam(lr_frozen),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
    ]

    t0 = time.time()
    hist_frozen = model.fit(train_ds, validation_data=val_ds, epochs=epochs_frozen,
                            callbacks=callbacks, verbose=1)
    frozen_time = time.time() - t0
    frozen_best_val = float(np.max(hist_frozen.history["val_accuracy"]))

    n = len(base.layers)
    unfreeze_from = int(n * (1 - unfreeze_last_fraction))
    for i, layer in enumerate(base.layers):
        layer.trainable = (i >= unfreeze_from)

    model.compile(
        optimizer=keras.optimizers.Adam(lr_ft),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    t1 = time.time()
    hist_ft = model.fit(train_ds, validation_data=val_ds, epochs=epochs_ft,
                        callbacks=callbacks, verbose=1)
    ft_time = time.time() - t1
    ft_best_val = float(np.max(hist_ft.history["val_accuracy"]))

    return model, {
        "model": model_name,
        "frozen_best_val_acc": round(frozen_best_val, 4),
        "finetune_best_val_acc": round(ft_best_val, 4),
        "unfreeze_from_layer": unfreeze_from,
        "frozen_time_sec": round(frozen_time, 1),
        "finetune_time_sec": round(ft_time, 1),
    }


In [ ]:
import pandas as pd
import os

MODEL_SPECS = [
    ("ResNet50", tf.keras.applications.ResNet50, tf.keras.applications.resnet50.preprocess_input),
    ("MobileNetV2", tf.keras.applications.MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input),
    ("InceptionV3", tf.keras.applications.InceptionV3, tf.keras.applications.inception_v3.preprocess_input),
]

OUT_DIR = "/content/task4_2_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

results = []

for name, BaseClass, preprocess_fn in MODEL_SPECS:
    print("\n==============================")
    print("Training:", name)
    print("==============================")

    model, row = train_and_finetune(
        model_name=name,
        BaseClass=BaseClass,
        preprocess_fn=preprocess_fn,
        epochs_frozen=5,
        epochs_ft=5,
        lr_frozen=1e-3,
        lr_ft=1e-5,
        unfreeze_last_fraction=0.2
    )

    model_path = os.path.join(OUT_DIR, f"{name}_finetuned.keras")
    model.save(model_path)
    row["saved_model"] = model_path

    results.append(row)

results_df = pd.DataFrame(results).sort_values("finetune_best_val_acc", ascending=False)
csv_path = os.path.join(OUT_DIR, "task4_2_results.csv")
results_df.to_csv(csv_path, index=False)

print("\nSaved results to:", csv_path)
results_df



Training: ResNet50
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 25s 157ms/step - accuracy: 0.8466 - loss: 0.4723 - val_accuracy: 0.9798 - val_loss: 0.0577
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 89ms/step - accuracy: 0.9676 - loss: 0.0935 - val_accuracy: 0.9731 - val_loss: 0.0632
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 89ms/step - accuracy: 0.9788 - loss: 0.0654 - val_accuracy: 0.9764 - val_loss: 0.0624
Epoch 4/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 94ms/step - accuracy: 0.9770 - loss: 0.0497 - val_accuracy: 0.9899 - val_loss: 0.0350
Epoch 5/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 93ms/step - accuracy: 0.9855 - loss: 0.0343 - val_accuracy: 0.9933 - val_loss: 0.0141
Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 31s 185ms/step - accuracy: 0.9736 - loss: 0.0894 - val_accuracy: 1.0000 - val_loss: 0.0096
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 134ms/step - accuracy: 0.9876 - loss: 0.0415 - val_accuracy: 1.0000 - val_loss: 0.0096
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 

,model,frozen_best_val_acc,finetune_best_val_acc,unfreeze_from_layer,frozen_time_sec,finetune_time_sec,saved_model
0,ResNet50,0.9933,1.0000,140,52.9,61.4,/content/task4_2_outputs/ResNet50_finetuned.keras
1,MobileNetV2,0.9899,0.9933,123,24.2,33.7,/content/task4_2_outputs/MobileNetV2_finetuned...
2,InceptionV3,0.9697,0.9933,248,35.5,60.0,/content/task4_2_outputs/InceptionV3_finetuned...
